In [ ]:
import geopandas as gpd
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point
from shapely.geometry.polygon import Polygon
from matplotlib.colors import ListedColormap


# **Conexión a google drive**
Para que funcione en tu drive los archivos deben estar en

`Mi unidad > eco2026`

In [ ]:
from google.colab import drive
import os
import pandas as pd

def load_drive_file(
    filename: str,
    folder_id: str = 'eco2026',
    mount_point: str = '/content/drive'
):
    """
    Monta Google Drive y carga un archivo desde una carpeta específica.

    Args:
        filename:    Nombre del archivo con extensión. Ej: 'Divipola_renamed.csv'
        folder_id:   Nombre de la carpeta en MyDrive. Default: 'eco2026'
        mount_point: Punto de montaje del drive. Default: '/content/drive'

    Returns:
        DataFrame con el contenido del archivo, o None si hubo un error.
    """

    # Montar el drive si no está montado aún
    if not os.path.exists(mount_point):
        print(f"Montando Google Drive en {mount_point}...")
        drive.mount(mount_point)
    else:
        print(f"Google Drive ya está montado en {mount_point}")

    # Construir la ruta completa
    folder_route = os.path.join(mount_point, 'MyDrive', folder_id)
    file_route = os.path.join(folder_route, filename)

    # Verificar que la carpeta existe
    if not os.path.exists(folder_route):
        print(f"❌ La carpeta '{folder_route}' no existe.")
        print(f"Carpetas disponibles en MyDrive: {os.listdir(os.path.join(mount_point, 'MyDrive'))}")
        return None

    # Verificar que el archivo existe
    if not os.path.exists(file_route):
        print(f"❌ El archivo '{filename}' no existe en '{folder_route}'.")
        print(f"Archivos disponibles: {os.listdir(folder_route)}")
        return None

    # Cargar según la extensión
    ext = filename.split('.')[-1].lower()
    loaders = {
        'csv':     lambda f: pd.read_csv(f),
        'xlsx':    lambda f: pd.read_excel(f),
        'xls':     lambda f: pd.read_excel(f),
        'json':    lambda f: pd.read_json(f),
        'parquet': lambda f: pd.read_parquet(f),
    }

    if ext not in loaders:
        print(f"❌ Extensión '.{ext}' no soportada. Extensiones válidas: {list(loaders.keys())}")
        return None

    df = loaders[ext](file_route)
    print(f"✅ '{filename}' cargado correctamente — {df.shape[0]} filas x {df.shape[1]} columnas")
    return df

# Renombrar, limpiar y ajustar los nombres de columnas

Los nombres de las columnas pueden venir en lenguaje natural, para ello se crea esta función que formatea correctamente los nombres

In [ ]:
import re

def clean_column_name(col):
    col = col.lower()                          # minúsculas
    col = col.strip()                          # quitar espacios al inicio/fin
    col = re.sub(r'[áàäâ]', 'a', col)         # normalizar tildes
    col = re.sub(r'[éèëê]', 'e', col)
    col = re.sub(r'[íìïî]', 'i', col)
    col = re.sub(r'[óòöô]', 'o', col)
    col = re.sub(r'[úùüû]', 'u', col)
    col = re.sub(r'ñ', 'n', col)
    col = re.sub(r'[^a-z0-9]+', '_', col)     # reemplazar todo lo demás con _
    col = re.sub(r'_+', '_', col)             # colapsar _ múltiples
    col = col.strip('_')                       # quitar _ al inicio/fin
    return col


In [ ]:
divipola_df = load_drive_file('DIVIPOLA_codigos_municipios.csv')
divipola_df.head(5)

In [ ]:
divipola_df.shape

In [ ]:
divipola_df.dtypes

# Limpia los nombres de columna de `divipola_df`

In [ ]:
divipola_df.columns = [clean_column_name(c) for c in divipola_df.columns]
print(divipola_df.columns.tolist())

# Encoding Ordinal de Variables Categóricas

Importa OrdinalEncoder de scikit-learn para transformar variables categóricas en valores numéricos enteros manteniendo un orden relativo entre categorías, preparando los datos para su uso en modelos de machine learning.

In [ ]:
tipo_municipio = divipola_df[["tipo_municipio_isla_area_no_municipalizada"]]

from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()
tipo_municipio_encoded = ordinal_encoder.fit_transform(tipo_municipio)

print(tipo_municipio_encoded)
divipola_df['tipo_municipio_encoded'] = tipo_municipio_encoded

In [ ]:
from sklearn.preprocessing import OneHotEncoder

tm_encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"

)

tipo_municipio = divipola_df[["tipo_municipio_isla_area_no_municipalizada"]]

tipo_municipio_hot = tm_encoder.fit_transform(tipo_municipio)


tipo_municipio_encoded = pd.DataFrame(
    tipo_municipio_hot,
    columns=tm_encoder.get_feature_names_out(),
    index=tipo_municipio.index
)


# Crear divipola_df_encoded: quitar la columna original y agregar las nuevas
divipola_encoded_df = pd.concat([
    divipola_df,
    #divipola_df.drop(columns=['tipo_municipio_isla_area_no_municipalizada']),
    tipo_municipio_encoded
], axis=1)


divipola_encoded_df.columns = [clean_column_name(c) for c in divipola_encoded_df.columns]
print(divipola_encoded_df.columns.tolist())

divipola_encoded_df



In [ ]:
divipola_encoded_df.dtypes

In [ ]:
divipola_encoded_df.head(5)

# Conversion y homogenizacion de datos
Normalizacion de nombres de columnas y tipos de datos

In [ ]:
eva_agro_df = load_drive_file('EvaluacionAgro_EVA_2026.csv')
# divipola_mncpios_df = load_drive_file('DIVIPOLA_codigos_municipios.csv')


In [ ]:
eva_agro_df.columns = [clean_column_name(c) for c in eva_agro_df.columns]
eva_agro_df.info()

In [ ]:
eva_agro_df

In [ ]:

eva_agro_df.info()

In [ ]:
eva_agro_df.head()

## Validacion de existencia de municipios

Se valida que todos los municipios reportados en los cultivos, existe en la base maestra de la divipola.

In [ ]:
#eva_agro_df['cod_mun'] = pd.to_numeric(eva_agro_df['cod_mun'], errors='coerce').astype('Int64')
eva_agro_df['cod_mun'] = (
    eva_agro_df['cod_mun']
    .astype(str)
    .str.replace(',', '', regex=False)  # eliminar coma separadora
    .str.strip()                         # quitar espacios
    .pipe(pd.to_numeric, errors='coerce')
    .astype('Int64')
)

# Verificar
print(eva_agro_df['cod_mun'].head(10).tolist())
print(eva_agro_df['cod_mun'].isna().sum(), "nulos después de conversión")
divipola_df['codigo_municipio'] = pd.to_numeric(divipola_df['codigo_municipio'], errors='coerce').astype('Int64')

# eva_agro_df['cod_mun']
# divipola_mncpios_df['codigo_municipio']

missing_municipios = eva_agro_df[~eva_agro_df['cod_mun'].isin(divipola_df['codigo_municipio'])]

print('missing_municipios')
missing_municipios
# Normalizar ambos a string con cero padding si es necesario
# eva_agro_df['cod_mun'] = eva_agro_df['cod_mun'].astype(str).str.zfill(5)
# divipola_mncpios_df['codigo_municipio'] = divipola_mncpios_df['codigo_municipio'].astype(str).str.zfill(5)

#missing_municipios = eva_agro_df[~eva_agro_df['cod_mun'].isin(divipola_mncpios_df['codigo_municipio'])]
#print(len(missing_municipios))

# print(f"Municipios en eva_agro no encontrados en divipola: {len(missing_municipios)}")
# print(missing_municipios['cod_mun'].unique())
# missing_municipios = eva_agro_df[~eva_agro_df['cod_mun'].isin(divipola_mncpios_df['codigo_municipio'])]

# eva_agro_df
# divipola_encoded_df
# missing_municipios

# unique_missing_municipios = missing_municipios['cod_mun'].unique()

# if len(unique_missing_municipios) > 0:
   #  print(f"Se encontro {len(unique_missing_municipios)}  cod_mcpio en  agro_df que no esta en divipola_df:\n{unique_missing_municipios}")
# else:
   #  print("todos los valores de  cod_mcpio values en agro_df estan presentes en divipola_df.")

In [ ]:
print("=== eva_agro cod_mun ===")
print("dtype:", eva_agro_df['cod_mun'].dtype)
print("muestra:", eva_agro_df['cod_mun'].head(5).tolist())
print("únicos:", eva_agro_df['cod_mun'].nunique())
print("nulos:", eva_agro_df['cod_mun'].isna().sum())

print("\n=== divipola codigo_municipio ===")
print("dtype:", divipola_df['codigo_municipio'].dtype)
print("muestra:", divipola_df['codigo_municipio'].head(5).tolist())
print("únicos:", divipola_df['codigo_municipio'].nunique())
print("nulos:", divipola_df['codigo_municipio'].isna().sum())

print("\n=== Intersección ===")
comunes = set(eva_agro_df['cod_mun']).intersection(set(divipola_df['codigo_municipio']))
print("Municipios en común:", len(comunes))
print("Muestra comunes:", list(comunes)[:5])

In [ ]:
missing_municipios = eva_agro_df[~eva_agro_df['cod_mun'].isin(divipola_df['codigo_municipio'])]

eva_agro_df
divipola_df
missing_municipios

unique_missing_municipios = missing_municipios['cod_mun'].unique()

if len(unique_missing_municipios) > 0:
    print(f"Se encontro {len(unique_missing_municipios)}  cod_mcpio en  agro_df que no esta en divipola_df:\n{unique_missing_municipios}")
else:
    print("todos los valores de  cod_mcpio values en agro_df estan presentes en divipola_df.")

## Analisis de rendimiento de  cultivos
Analizaremos los 10 cultivos que tienen mayor rendimiento,  y los municipios donde presentan este comportamiento.

In [ ]:
eva_agro_df

In [ ]:

eva_agro_df.dtypes

In [ ]:
def parsear_columnas_a_numero(df, columnas, tipo='float64'):
    df_copia = df.copy()

    for col in columnas:
        if col not in df_copia.columns:
            print(f"❌ Columna '{col}' no encontrada")
            continue

        print(f"\n🔄 Procesando: {col}")

        # PASO 1: Limpiar valores
        cleaned = (
            df_copia[col]
            .astype(str)
            .str.strip()
            .str.replace(',', '.', regex=False)
            .str.replace(' ', '', regex=False)
            .str.replace('$', '', regex=False)
            .str.replace('%', '', regex=False)
        )

        print(f"   Después de limpiar (primeros 5): {cleaned.head().tolist()}")

        # PASO 2: Convertir a número
        numeric = pd.to_numeric(cleaned, errors='coerce')
        print(f"   Después de to_numeric (dtype: {numeric.dtype})")
        print(f"   NaN detectados: {numeric.isna().sum()}")

        # PASO 3: Convertir a Int64
        try:
            df_copia[col] = numeric.astype(tipo)
            print(f"   ✅ Conversión exitosa → {tipo}")
        except Exception as e:
            print(f"   ❌ Error: {e}")
            print(f"   Valores no convertibles: {df_copia[numeric.isna()][col].unique()[:5]}")

    return df_copia


# USO:
# agro_df = parsear_columnas_a_numero(agro_df, ['AÑO', 'PRECIO', 'CANTIDAD'])

# Verificar tipos
# print(agro_df.dtypes)
# USO:
# agro_df = parsear_columnas_a_numero(agro_df, ['AÑO', 'PRECIO', 'CANTIDAD'])

In [ ]:
eva_agro_df = parsear_columnas_a_numero(eva_agro_df, ['ano', 'area_sembrada_ha', 'area_cosechada_ha', 'produccion_t'])

In [ ]:
eva_agro_df.dtypes

In [ ]:
# Group by MUNICIPIO and CULTIVO and calculate the mean rendimiento_t_ha
yield_performance = eva_agro_df.groupby(['municipio', 'cultivo'])['rendimiento_t_ha'].mean().reset_index()

# Sort by rendimiento_t_ha in descending order and get the top 20
top_20_yield = yield_performance.sort_values(by='rendimiento_t_ha', ascending=False).head(20)

print("Top 20 municipalities and crops by yield performance:")
print(top_20_yield)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calcular el rendimiento promedio por cultivo
rendimiento_por_cultivo = eva_agro_df.groupby('cultivo')['rendimiento_t_ha'].mean().reset_index()

# Obtener los 10 cultivos con mayor rendimiento
top_10_cultivos = rendimiento_por_cultivo.nlargest(10, 'rendimiento_t_ha')

# Crear el gráfico de barras
plt.figure(figsize=(12, 7))
sns.barplot(x='rendimiento_t_ha', y='cultivo', data=top_10_cultivos, palette='viridis', hue='cultivo', legend=False)
plt.title('Top 10 Cultivos por Rendimiento Promedio (t/ha)')
plt.xlabel('Rendimiento Promedio (t/ha)')
plt.ylabel('Cultivo')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

## Conteo de municipios con cultivos por departamento

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Agrupar por departamento y contar el número de municipios únicos con cultivos
municipios_por_departamento = eva_agro_df.groupby('departamento')['cod_mun'].nunique().reset_index()
municipios_por_departamento = municipios_por_departamento.sort_values(by='cod_mun', ascending=False)

# Crear el gráfico de barras
plt.figure(figsize=(15, 8))
sns.barplot(x='cod_mun', y='departamento', data=municipios_por_departamento, palette='viridis', hue='departamento', legend=False)
plt.title('Número de Municipios con Cultivos por Departamento')
plt.xlabel('Número de Municipios con Cultivos')
plt.ylabel('Departamento')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

## Análisis del Grupo de Cultivo y Cultivo con Más Registros

In [ ]:
eva_agro_df

In [ ]:
# Contar la frecuencia de cada combinación de grupo_cultivo y CULTIVO
conteo_cultivos = eva_agro_df.groupby(['grupo_de_cultivo', 'cultivo']).size().reset_index(name='cantidad_registros')

# Encontrar el grupo_cultivo y CULTIVO con la mayor cantidad de registros
cultivo_mas_frecuente = conteo_cultivos.loc[conteo_cultivos['cantidad_registros'].idxmax()]

print("El grupo de cultivo y cultivo con más registros es:")
print(cultivo_mas_frecuente)

top_grupo_cultivo = cultivo_mas_frecuente['grupo_de_cultivo']
top_cultivo = cultivo_mas_frecuente['cultivo']


### Departamentos con más registros para el cultivo más frecuente

In [ ]:
# Filtrar el DataFrame por el grupo_cultivo y CULTIVO más frecuente
df_top_cultivo = eva_agro_df[(eva_agro_df['grupo_de_cultivo'] == top_grupo_cultivo) & (eva_agro_df['cultivo'] == top_cultivo)]

# Contar los registros por DEPARTAMENTO para este cultivo
registros_por_departamento_top_cultivo = df_top_cultivo['departamento'].value_counts().reset_index()
registros_por_departamento_top_cultivo.columns = ['departamento', 'cantidad_registros']

# Ordenar para visualización
registros_por_departamento_top_cultivo = registros_por_departamento_top_cultivo.sort_values(by='cantidad_registros', ascending=False)

print(f"Departamentos con más registros para el grupo de cultivo '{top_grupo_cultivo}' y cultivo '{top_cultivo}':")
print(registros_por_departamento_top_cultivo.head(10))

# Crear el gráfico de barras
plt.figure(figsize=(14, 8))
sns.barplot(x='cantidad_registros', y='departamento', data=registros_por_departamento_top_cultivo.head(10), palette='plasma', hue='departamento', legend=False)
plt.title(f'Top 10 Departamentos con más Registros de {top_grupo_cultivo} - {top_cultivo}')
plt.xlabel('Cantidad de Registros')
plt.ylabel('Departamento')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Análisis de Cultivos con Área Cosechada >= Área Sembrada

## Análisis de Cultivos con Mayor Área Sembrada y Peor Relación Cosecha/Siembra (Revisado)

Esta gráfica muestra los 10 cultivos con la mayor **Área Sembrada Total (ha)** que tienen una **relación de cosecha sobre siembra menor a 1**. Esto indica cultivos donde una parte significativa del área sembrada no fue cosechada, sugiriendo una posible ineficiencia.

*   El **eje X** representa la `Área Sembrada Total (ha)` en millones.
*   Para cada barra, se mostrarán **dos valores clave**:
    *   El **Área Sembrada Total** dentro o cerca de la barra.
    *   La **Relación Promedio Cosecha/Siembra** (`Ratio`) de forma explícita al final de cada barra.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filtrar filas donde el área sembrada no es nula y no es cero, y el área cosechada no es nula
df_filtered = eva_agro_df[eva_agro_df['area_sembrada_ha'].notna() & (eva_agro_df['area_sembrada_ha'] > 0) & eva_agro_df['area_cosechada_ha'].notna()].copy()

# Calcular la relación de área cosechada sobre área sembrada
df_filtered['relacion_cosecha_sembrada'] = df_filtered['area_cosechada_ha'] / df_filtered['area_sembrada_ha']

# Filtrar los cultivos donde la relación es mayor o igual a 1
cultivos_eficientes = df_filtered[df_filtered['relacion_cosecha_sembrada'] >= 1]

# Calcular el promedio de la relación por CULTIVO
rendimiento_por_cultivo_eficiente = cultivos_eficientes.groupby('cultivo')['relacion_cosecha_sembrada'].mean().reset_index()

# Obtener los 10 cultivos con la mejor relación promedio
top_10_relacion_cultivos = rendimiento_por_cultivo_eficiente.nlargest(10, 'relacion_cosecha_sembrada')

print("Top 10 Cultivos con la mejor relación Área Cosechada / Área Sembrada (>= 1):")
print(top_10_relacion_cultivos)

# Crear el gráfico de barras
plt.figure(figsize=(12, 7))
sns.barplot(x='relacion_cosecha_sembrada', y='cultivo', data=top_10_relacion_cultivos, palette='magma', hue='cultivo', legend=False)
plt.title('Top 10 Cultivos por Relación Promedio (Área Cosechada / Área Sembrada >= 1)')
plt.xlabel('Relación Promedio (Área Cosechada / Área Sembrada)')
plt.ylabel('Cultivo')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
import requests
import pandas as pd

def consultar_estaciones_ideam(fecha_inicio, fecha_fin):
    """
    Consulta datos de estaciones IDEAM en datos.gov.co

    Args:
        fecha_inicio: str 'YYYY-MM-DD' o 'YYYY-MM-DDTHH:MM:SS'
        fecha_fin: str 'YYYY-MM-DD' o 'YYYY-MM-DDTHH:MM:SS'

    Returns:
        DataFrame (vacío si no hay datos)
    """
    # Agregar hora si no la tiene
    if 'T' not in fecha_inicio:
        fecha_inicio = f"{fecha_inicio}T00:00:00"
    if 'T' not in fecha_fin:
        fecha_fin = f"{fecha_fin}T23:59:59"

    dataset_id = "57sv-p2fu"
    url = f"https://www.datos.gov.co/resource/{dataset_id}.json"

    # Query con backticks y casting :: floating_timestamp
    query = f"""
    SELECT
      `codigoestacion`,
      `codigosensor`,
      `fechaobservacion`,
      `valorobservado`,
      `nombreestacion`,
      `departamento`,
      `municipio`,
      `zonahidrografica`,
      `latitud`,
      `longitud`,
      `descripcionsensor`,
      `unidadmedida`,
      `entidad`
    WHERE
      `fechaobservacion`
        BETWEEN "{fecha_inicio}" :: floating_timestamp
        AND "{fecha_fin}" :: floating_timestamp
    LIMIT 50000
    """

    params = {'$query': query}

    print(f"📡 Consultando datos entre {fecha_inicio} y {fecha_fin}...")

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()

        data = response.json()

        # Columnas esperadas
        columnas = [
            'codigoestacion', 'codigosensor', 'fechaobservacion', 'valorobservado',
            'nombreestacion', 'departamento', 'municipio', 'zonahidrografica',
            'latitud', 'longitud', 'descripcionsensor', 'unidadmedida', 'entidad'
        ]

        if len(data) == 0:
            print(f"✅ 0 registros encontrados")
            return pd.DataFrame(columns=columnas)

        df = pd.DataFrame(data)

        # Conversiones de tipos
        df['fechaobservacion'] = pd.to_datetime(df['fechaobservacion'])
        df['valorobservado'] = pd.to_numeric(df['valorobservado'], errors='coerce')
        df['latitud'] = pd.to_numeric(df['latitud'], errors='coerce')
        df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')

        print(f"✅ Se obtuvieron {len(df)} registros")
        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return pd.DataFrame()


# USO:
# agro_df = consultar_estaciones_ideam('2026-06-01', '2026-06-03')
# print(agro_df.head())

In [ ]:
agro_df = consultar_estaciones_ideam('2026-06-01', '2026-06-03')
print(agro_df.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Agrupar por CULTIVO para obtener el área sembrada total y el rendimiento promedio
analisis_area_rendimiento = df_filtered.groupby('cultivo').agg(
    total_area_sembrada_ha=('area_sembrada_ha', 'sum'),
    promedio_rendimiento_t_ha=('rendimiento_t_ha', 'mean')
).reset_index()

# Filtrar cultivos con rendimiento promedio mayor a cero para evitar divisiones por cero o infinitos
analisis_area_rendimiento = analisis_area_rendimiento[analisis_area_rendimiento['promedio_rendimiento_t_ha'] > 0]

# Calcular la nueva métrica de 'ineficiencia': Área Sembrada Total / Rendimiento Promedio
# Un valor más alto indica que se necesita más área para la misma cantidad de rendimiento.
analisis_area_rendimiento['inefficiency_score'] = analisis_area_rendimiento['total_area_sembrada_ha'] / analisis_area_rendimiento['promedio_rendimiento_t_ha']

# Eliminar cualquier NaN que pueda haber surgido de cálculos
analisis_area_rendimiento = analisis_area_rendimiento.dropna(subset=['inefficiency_score'])

# Ordenar por 'inefficiency_score' en orden descendente para obtener los 10 más ineficientes
top_10_inefficient_crops = analisis_area_rendimiento.sort_values(
    by='inefficiency_score', ascending=False
).head(10)

print("Top 10 Cultivos con mayor Ineficiencia (Área Sembrada Total / Rendimiento Promedio):\n")
print(top_10_inefficient_crops)

# --- Graficar los resultados ---
plt.figure(figsize=(15, 9))

sns.barplot(x='inefficiency_score', y='cultivo', data=top_10_inefficient_crops, palette='viridis', hue='cultivo', legend=False)
plt.title('Top 10 Cultivos con Mayor Ineficiencia (Área Sembrada / Rendimiento)', fontsize=16)
plt.xlabel('Relación (Área Sembrada Total / Rendimiento Promedio) - Mayor es Peor', fontsize=12)
plt.ylabel('Cultivo', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)

# Añadir etiquetas de texto para el score de ineficiencia (dentro de la barra),
# el área sembrada y el rendimiento promedio (fuera de la barra)
for plot_idx, (df_idx, row) in enumerate(top_10_inefficient_crops.iterrows()):
    # Posición en el eje X (final de la barra)
    x_bar_end = row['inefficiency_score']
    # Posición en el eje Y (índice de la barra en el gráfico)
    y_pos = plot_idx

    # Texto para el score de ineficiencia (dentro de la barra, ligeramente a la izquierda del final)
    plt.text(x_bar_end * 0.95, y_pos, f'{x_bar_end:,.0f}',
             color='white', ha='right', va='center', fontsize=9, weight='bold')

    # Offset para posicionar los textos fuera de la barra de manera consistente
    max_score = top_10_inefficient_crops['inefficiency_score'].max()
    offset_from_bar = max_score * 0.02

    # Texto del área sembrada (fuera de la barra, ligeramente por debajo del centro)
    plt.text(x_bar_end + offset_from_bar, y_pos - 0.2,
             f'Área: {row['total_area_sembrada_ha'] / 1_000_000:.2f} M ha',
             color='black', ha='left', va='center', fontsize=9)

    # Texto del rendimiento promedio (fuera de la barra, ligeramente por encima del centro)
    plt.text(x_bar_end + offset_from_bar, y_pos + 0.2,
             f'Rend: {row['promedio_rendimiento_t_ha']:.2f} t/ha',
             color='dimgray', ha='left', va='center', fontsize=9)

plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout() # Ajustar layout para prevenir solapamiento
plt.show()

Only test github connection